In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from pathlib import Path

REVIEW_CSV   = Path('annotations/review_queue.csv')
LABELS_CSV   = Path('annotations/labels_clean.csv')
CLEANED_DIR  = Path('data/cleaned')

df = pd.read_csv(REVIEW_CSV)
# Only show clips not yet labelled
remaining = df[df['your_label'].isna() | (df['your_label'] == '')].copy()
total = len(remaining)
indices = list(remaining.index)
pos = [0]  # mutable pointer

print(f'{total} clips to review  ({len(df) - total} already done)')

103 clips to review  (0 already done)


In [ ]:
# ── UI layout ──────────────────────────────────────────────────────────────

progress_bar  = widgets.IntProgress(value=0, min=0, max=total, description='Progress:', bar_style='info', layout=widgets.Layout(width='100%'))
progress_label = widgets.Label(value='')

clip_id_label  = widgets.HTML(value='')
auto_label_box = widgets.HTML(value='')
transcript_box = widgets.Textarea(value='', disabled=True,
                                   layout=widgets.Layout(width='100%', height='140px'))
audio_out      = widgets.Output()

btn_calm   = widgets.Button(description='Calm',        button_style='success', layout=widgets.Layout(width='160px', height='45px'))
btn_frust  = widgets.Button(description='Frustrated',  button_style='warning', layout=widgets.Layout(width='160px', height='45px'))
btn_stress = widgets.Button(description='High Stress', button_style='danger',  layout=widgets.Layout(width='160px', height='45px'))
btn_skip   = widgets.Button(description='Skip',        button_style='',        layout=widgets.Layout(width='100px', height='45px'))
btn_prev   = widgets.Button(description='< Back',      button_style='',        layout=widgets.Layout(width='100px', height='45px'))

status_out = widgets.Output()

def colour_label(lbl):
    colours = {'Calm': 'green', 'Frustrated': 'darkorange', 'High Stress': 'crimson'}
    c = colours.get(lbl, 'grey')
    return f'<b style="color:{c}">{lbl}</b>'

def load_clip(i):
    if i >= len(indices):
        clip_id_label.value  = '<h3>All done!</h3>'
        auto_label_box.value = ''
        transcript_box.value = 'All clips have been labelled. Run the save cell below.'
        audio_out.clear_output()
        for b in [btn_calm, btn_frust, btn_stress, btn_skip]:
            b.disabled = True
        return

    row = df.loc[indices[i]]
    done = sum(1 for idx in indices[:i] if df.loc[idx, 'your_label'] not in ('', None))

    progress_bar.value  = done
    progress_label.value = f'{done} / {total} labelled'

    your = df.loc[indices[i], 'your_label']
    your_str = f'  |  <b>Your label:</b> {colour_label(str(your))}' if your not in ('', None, float('nan')) else ''

    clip_id_label.value  = f'<h3>Clip {i+1} / {total} &nbsp;&nbsp; <code>{row["clip_id"]}</code>{your_str}</h3>'
    auto_label_box.value = (f'<b>Auto:</b> {colour_label(str(row["auto_label"]))} '
                             f'(conf {row["confidence"]:.2f}) &nbsp;|&nbsp; '
                             f'Text model: {colour_label(str(row["text_model_label"]))} ({row["text_conf"]:.2f}) &nbsp;|&nbsp; '
                             f'Acoustic: {colour_label(str(row["acoustic_label"]))} ({row["acoustic_conf"]:.2f})')
    transcript_box.value = str(row['transcript']) if pd.notna(row['transcript']) else '(no transcript)'

    audio_out.clear_output()
    wav = CLEANED_DIR / f"{row['clip_id']}.wav"
    with audio_out:
        if wav.exists():
            display(Audio(filename=str(wav), autoplay=False))
        else:
            print('(audio file not found)')

def apply_label(label):
    i = pos[0]
    if i >= len(indices):
        return
    df.loc[indices[i], 'your_label'] = label
    df.to_csv(REVIEW_CSV, index=False)
    with status_out:
        clear_output(wait=True)
        print(f'Saved: {df.loc[indices[i], "clip_id"]} -> {label}')
    pos[0] += 1
    load_clip(pos[0])

btn_calm.on_click(  lambda _: apply_label('Calm'))
btn_frust.on_click( lambda _: apply_label('Frustrated'))
btn_stress.on_click(lambda _: apply_label('High Stress'))
btn_skip.on_click(  lambda _: (pos.__setitem__(0, pos[0]+1), load_clip(pos[0])))

def go_back(_):
    if pos[0] > 0:
        pos[0] -= 1
        load_clip(pos[0])
btn_prev.on_click(go_back)

ui = widgets.VBox([
    widgets.HBox([progress_bar, progress_label]),
    clip_id_label,
    auto_label_box,
    transcript_box,
    audio_out,
    widgets.HBox([btn_calm, btn_frust, btn_stress, btn_skip, btn_prev]),
    status_out,
])

display(ui)
load_clip(pos[0])

In [3]:
# ── Merge your labels back into labels_clean.csv ───────────────────────────
# Run this cell when you are done reviewing.

df = pd.read_csv(REVIEW_CSV)
labelled = df[df['your_label'].notna() & (df['your_label'] != '')]
skipped  = df[df['your_label'].isna()  | (df['your_label'] == '')]

print(f'Labelled : {len(labelled)}')
print(f'Skipped  : {len(skipped)}')
if len(skipped):
    print(f'Skipped clips will keep their auto_label.')

labels_df = pd.read_csv(LABELS_CSV)

# Apply your labels; fallback to auto_label for skipped
review_map = {}
for _, row in df.iterrows():
    lbl = row['your_label'] if (pd.notna(row['your_label']) and row['your_label'] != '') else row['auto_label']
    review_map[row['clip_id']] = lbl

labels_df['final_label'] = labels_df.apply(
    lambda r: review_map.get(r['clip_id'], r['final_label']), axis=1
)
labels_df['flagged_for_review'] = False  # all reviewed now

labels_df.to_csv(LABELS_CSV, index=False)
print(f'\nSaved updated labels to {LABELS_CSV}')
print()
print('Final label distribution:')
print(labels_df['final_label'].value_counts().to_string())

Labelled : 103
Skipped  : 0

Saved updated labels to annotations\labels_clean.csv

Final label distribution:
final_label
Calm           863
High Stress    296
Frustrated     147
